In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/xyz2005/notebook-7/report_generation_results.csv
/kaggle/input/datasets/xyz2005/notebook-7/generated_reports.csv


In [2]:
!pip install -q nltk rouge-score bert-score

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 2.2 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

print("Evaluation packages loaded successfully.")

Evaluation packages loaded successfully.


In [4]:
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /usr/share/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /usr/share/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [5]:
import os

FILE_PATH = "/kaggle/input/datasets/xyz2005/notebook-7/generated_reports.csv"

print("File exists:", os.path.exists(FILE_PATH))

File exists: True


In [6]:
df = pd.read_csv(FILE_PATH)

print("Number of reports:", len(df))
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst row:")
display(df.iloc[0])

Number of reports: 496

Columns:
['report', 'Pneumonia', 'Cardiomegaly', 'Pleural Effusion', 'Atelectasis', 'Edema', 'Pneumothorax', 'Consolidation', 'Lung Opacity', 'Nodule', 'Mass', 'generated_report']

First row:


report              stable appearance of hiatal hernia. clear righ...
Pneumonia                                                           0
Cardiomegaly                                                        0
Pleural Effusion                                                    1
Atelectasis                                                         0
Edema                                                               0
Pneumothorax                                                        1
Consolidation                                                       0
Lung Opacity                                                        0
Nodule                                                              0
Mass                                                                1
generated_report    the heart size remains within normal limits. t...
Name: 0, dtype: object

In [7]:
print(df.isnull().sum())

report              0
Pneumonia           0
Cardiomegaly        0
Pleural Effusion    0
Atelectasis         0
Edema               0
Pneumothorax        0
Consolidation       0
Lung Opacity        0
Nodule              0
Mass                0
generated_report    0
dtype: int64


In [8]:
df = df.dropna(
    subset=[
        "report",
        "generated_report"
    ]
).reset_index(drop=True)

print("Reports available for evaluation:", len(df))

Reports available for evaluation: 496


In [9]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"\s+", " ", text)

    text = text.strip()

    return text

In [10]:
df["reference"] = df["report"].apply(clean_text)
df["prediction"] = df["generated_report"].apply(clean_text)

print("Cleaning completed.")

Cleaning completed.


In [11]:
smooth = SmoothingFunction().method1

bleu_scores = []

for reference, prediction in zip(
    df["reference"],
    df["prediction"]
):

    reference_tokens = reference.split()
    prediction_tokens = prediction.split()

    if len(prediction_tokens) == 0:
        bleu = 0.0
    else:
        bleu = sentence_bleu(
            [reference_tokens],
            prediction_tokens,
            smoothing_function=smooth
        )

    bleu_scores.append(bleu)

df["BLEU"] = bleu_scores

mean_bleu = np.mean(bleu_scores)

print(f"Average BLEU: {mean_bleu:.4f}")

Average BLEU: 0.0055


In [12]:
scorer = rouge_scorer.RougeScorer(
    ["rouge1", "rouge2", "rougeL"],
    use_stemmer=True
)

rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for reference, prediction in zip(
    df["reference"],
    df["prediction"]
):

    scores = scorer.score(
        reference,
        prediction
    )

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

In [13]:
mean_rouge1 = np.mean(rouge1_scores)
mean_rouge2 = np.mean(rouge2_scores)
mean_rougeL = np.mean(rougeL_scores)

print(f"ROUGE-1: {mean_rouge1:.4f}")
print(f"ROUGE-2: {mean_rouge2:.4f}")
print(f"ROUGE-L: {mean_rougeL:.4f}")

ROUGE-1: 0.1201
ROUGE-2: 0.0145
ROUGE-L: 0.0891


In [14]:
from bert_score import score

predictions = df["prediction"].tolist()
references = df["reference"].tolist()

P, R, F1 = score(
    predictions,
    references,
    lang="en",
    verbose=True
)

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

calculating scores...
computing bert embedding.


  0%|          | 0/14 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/8 [00:00<?, ?it/s]

done in 398.26 seconds, 1.25 sentences/sec


In [15]:
bert_precision = P.mean().item()
bert_recall = R.mean().item()
bert_f1 = F1.mean().item()

print(f"BERTScore Precision: {bert_precision:.4f}")
print(f"BERTScore Recall:    {bert_recall:.4f}")
print(f"BERTScore F1:        {bert_f1:.4f}")

BERTScore Precision: 0.8241
BERTScore Recall:    0.8467
BERTScore F1:        0.8351


In [16]:
results = pd.DataFrame({
    "Metric": [
        "BLEU",
        "ROUGE-1",
        "ROUGE-2",
        "ROUGE-L",
        "BERTScore Precision",
        "BERTScore Recall",
        "BERTScore F1"
    ],

    "Score": [
        mean_bleu,
        mean_rouge1,
        mean_rouge2,
        mean_rougeL,
        bert_precision,
        bert_recall,
        bert_f1
    ]
})

display(results)

,Metric,Score
0,BLEU,0.005525
1,ROUGE-1,0.120073
2,ROUGE-2,0.014487
3,ROUGE-L,0.089059
4,BERTScore Precision,0.824091
5,BERTScore Recall,0.846726
6,BERTScore F1,0.835137


In [17]:
results.to_csv(
    "/kaggle/working/evaluation_results_baseline.csv",
    index=False
)

print("Evaluation results saved.")

Evaluation results saved.


In [18]:
df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores
df["BERTScore-F1"] = F1.cpu().numpy()

df.to_csv(
    "/kaggle/working/detailed_evaluation_results.csv",
    index=False
)

print("Detailed evaluation saved.")

Detailed evaluation saved.


In [19]:
for i in range(min(10, len(df))):

    print("=" * 80)
    print(f"CASE {i+1}")

    print("\nFINDINGS:")
    print(df.iloc[i]["prediction"])

    print("\nGENERATED:")
    print(df.iloc[i]["generated_report"])

    print("\nGROUND TRUTH:")
    print(df.iloc[i]["report"])

    print("\nROUGE-L:")
    print(df.iloc[i]["ROUGE-L"])

    print()

CASE 1

FINDINGS:
the heart size remains within normal limits. the lungs remain grossly clear without evidence for acute infiltrate or significant change since comparison study was performed on xxxx. however, there appears to be some increase in prominence of the mediastinal contours bilaterally suggesting mild vascular congestion versus perihilar opacities consistent with emphysema. additionally, there may have been slight improvement in visualized hiatal hiatus hernia noted previously. otherwise, no definite metastatic disease identified. if clinically indicated consider

GENERATED:
the heart size remains within normal limits. the lungs remain grossly clear without evidence for acute infiltrate or significant change since comparison study was performed on xxxx. however, there appears to be some increase in prominence of the mediastinal contours bilaterally suggesting mild vascular congestion versus perihilar opacities consistent with emphysema. additionally, there may have been sligh

In [20]:
def repetition_ratio(text):

    tokens = text.split()

    if len(tokens) == 0:
        return 0

    unique_tokens = len(set(tokens))

    return 1 - (unique_tokens / len(tokens))

In [21]:
df["repetition_ratio"] = df["prediction"].apply(
    repetition_ratio
)

print(
    "Average repetition ratio:",
    df["repetition_ratio"].mean()
)

Average repetition ratio: 0.07398429443562411


In [22]:
df["generated_length"] = df["prediction"].apply(
    lambda x: len(x.split())
)

df["reference_length"] = df["reference"].apply(
    lambda x: len(x.split())
)

print(
    "Average generated length:",
    df["generated_length"].mean()
)

print(
    "Average reference length:",
    df["reference_length"].mean()
)

Average generated length: 73.16935483870968
Average reference length: 40.08467741935484


In [23]:
print(
    df[
        [
            "generated_length",
            "reference_length",
            "repetition_ratio"
        ]
    ].describe()
)

       generated_length  reference_length  repetition_ratio
count        496.000000        496.000000        496.000000
mean          73.169355         40.084677          0.073984
std            7.380807         19.766028          0.034420
min           12.000000         12.000000          0.000000
25%           70.000000         26.000000          0.051282
50%           73.000000         36.000000          0.071008
75%           77.250000         48.000000          0.097222
max           94.000000        129.000000          0.186667


In [24]:
df.to_csv(
    "/kaggle/working/final_baseline_evaluation.csv",
    index=False
)

print("Final baseline evaluation file saved!")

Final baseline evaluation file saved!
